# 01 – Ethical CDSS: Training Pipeline

Runs the full pipeline from raw CSVs to saved model artifacts.

**Output files written to `../data/`:**
- `kde_model.joblib` — fitted `EthicalKDEAnomalyDetector`
- `windows.parquet` — sliding-window feature matrix with pre-computed `anomaly_score`
- `prior_rules.json` — Italian clinical rules for the symbolic AI layer

Run cells top-to-bottom. If real CSVs are not yet placed in `../data/`, the notebook
falls back to a synthetic dataset so every cell can still be executed end-to-end.

In [ ]:
import json
import logging
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Add project root to path so util/ is importable from the notebooks/ directory.
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from util.data_processing import merge_er_data, build_features, create_sliding_windows
from util.models import EthicalKDEAnomalyDetector

logging.basicConfig(level=logging.INFO, format='%(levelname)s – %(message)s')
sns.set_theme(style='whitegrid')
%matplotlib inline

DATA_DIR = PROJECT_ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)
print(f'Project root : {PROJECT_ROOT}')
print(f'Data directory: {DATA_DIR}')

## Step 1 – Load Raw CSV Data

Expected files in `../data/`:
- `Accessi.csv` — master access registry (`ID_ACCESSO`, `ID_PAZIENTE`, `DATAORA_ACCETTAZIONE`, `DATA_NASCITA`, `SESSO`, …)
- `Clinici.csv` — clinical records (`ID_ACCESSO`, `Diagnosi`, `Sintomi`, …)
- `Triage.csv` — triage records (`ID_ACCESSO`, `Motivo_accesso`, `codice_triage`, …)

If any file is missing the notebook generates a synthetic dataset that matches the expected schema.

In [ ]:
def _make_synthetic_data(n_patients: int = 250, seed: int = 42) -> tuple:
    """Generate synthetic ER data matching the expected merged schema."""
    rng = np.random.default_rng(seed)
    base_date = pd.Timestamp('2022-01-01')

    accessi_rows, clinici_rows, triage_rows = [], [], []
    access_id = 1

    italian_diagnoses = [
        'Contusione', 'Frattura', 'Ustione', 'Lacerazione',
        'Trauma cranico', 'Ecchimosi', 'Lesione cutanea', 'Febbre',
    ]
    italian_symptoms = [
        'dolore addominale', 'fratture multiple', 'lesioni cutanee',
        'ustioni', 'ematomi', 'trauma cranico', 'ecchimosi diffuse',
        'tosse', 'febbre alta', 'vomito',
    ]
    motivi = [
        'caduta accidentale', 'trauma', 'ustione', 'ferita',
        'dolore', 'convulsione', 'difficolta respiratoria',
    ]

    for pid in range(1, n_patients + 1):
        n_visits = int(rng.choice([1, 2, 3, 4, 5, 6, 8], p=[0.25, 0.25, 0.2, 0.1, 0.1, 0.05, 0.05]))
        birth_date = base_date - pd.Timedelta(days=int(rng.integers(180, 6 * 365)))
        sex = rng.choice(['M', 'F'])
        nationality = rng.choice(['IT', 'IT', 'IT', 'EU', 'ExtraEU'], p=[0.5, 0.15, 0.15, 0.1, 0.1])

        visit_date = base_date + pd.Timedelta(days=int(rng.integers(0, 180)))
        for _ in range(n_visits):
            aid = access_id
            access_id += 1

            accessi_rows.append({
                'ID_ACCESSO': aid,
                'ID_PAZIENTE': pid,
                'DATAORA_ACCETTAZIONE': visit_date,
                'DATA_NASCITA': birth_date,
                'SESSO': sex,
                'NAZIONALITA': nationality,
            })

            # Abuse-like patients (pid % 20 == 0) have suggestive text.
            is_abuse_like = (pid % 20 == 0)
            diag = rng.choice(italian_diagnoses)
            symp = rng.choice(italian_symptoms)
            if is_abuse_like:
                symp = rng.choice(['fratture multiple', 'lesioni cutanee', 'ustioni', 'ematomi'])

            clinici_rows.append({
                'ID_ACCESSO': aid,
                'Diagnosi': diag,
                'Sintomi': symp,
                'codice_gravita': int(rng.integers(1, 5)),
            })

            triage_rows.append({
                'ID_ACCESSO': aid,
                'Motivo_accesso': rng.choice(motivi),
                'codice_triage': int(rng.integers(1, 6)),
            })

            gap_days = int(rng.integers(3, 45)) if not is_abuse_like else int(rng.integers(1, 10))
            visit_date = visit_date + pd.Timedelta(days=gap_days)

    return (
        pd.DataFrame(accessi_rows),
        pd.DataFrame(clinici_rows),
        pd.DataFrame(triage_rows),
    )


# --- Attempt to load real CSVs; fall back to synthetic if not found. --------
csv_paths = {
    'accessi': DATA_DIR / 'Accessi.csv',
    'clinici': DATA_DIR / 'Clinici.csv',
    'triage':  DATA_DIR / 'Triage.csv',
}

all_found = all(p.exists() for p in csv_paths.values())

if all_found:
    print('Real CSV files found — loading...')
    df_accessi = pd.read_csv(csv_paths['accessi'], low_memory=False)
    df_clinici = pd.read_csv(csv_paths['clinici'], low_memory=False)
    df_triage  = pd.read_csv(csv_paths['triage'],  low_memory=False)
    print(f'  Accessi : {df_accessi.shape}')
    print(f'  Clinici : {df_clinici.shape}')
    print(f'  Triage  : {df_triage.shape}')
else:
    missing = [str(p.name) for p in csv_paths.values() if not p.exists()]
    print(f'WARNING: CSV files not found ({missing}). Using synthetic dataset.')
    df_accessi, df_clinici, df_triage = _make_synthetic_data(n_patients=300)
    print(f'  Synthetic accessi : {df_accessi.shape}')
    print(f'  Synthetic clinici : {df_clinici.shape}')
    print(f'  Synthetic triage  : {df_triage.shape}')

## Step 2 – Merge and Feature Engineering

In [ ]:
# Merge the three tables.
df_merged = merge_er_data(df_accessi, df_clinici, df_triage)
print(f'Merged shape: {df_merged.shape}')
print(f'Columns: {list(df_merged.columns)}')

In [ ]:
# Engineer temporal and clinical features.
df_feat = build_features(df_merged)
print(f'Shape after feature engineering: {df_feat.shape}')

# Quick sanity check on the four new columns.
check_cols = ['age_months', 'days_since_last_visit', 'num_visits_90d', 'inter_visit_variance']
display(df_feat[check_cols].describe().round(2))

In [ ]:
# Visit frequency distribution (used to calibrate the normal/suspicious split).
visit_counts = df_feat.groupby('ID_PAZIENTE').size().rename('n_visits')
fig, ax = plt.subplots(figsize=(8, 3))
visit_counts.hist(bins=range(1, visit_counts.max() + 2), ax=ax, color='#3498db', edgecolor='white')
ax.set_xlabel('Visits per patient')
ax.set_ylabel('Count')
ax.set_title('Distribution of ER visit frequency per patient')
plt.tight_layout()
plt.show()

print(visit_counts.describe().to_string())

## Step 3 – Sliding Window Matrix

`numeric_only=False` is **mandatory** so that Italian text columns
(`Sintomi`, `Diagnosi`, `Motivo_accesso`) survive into the window and
are available for the symbolic-AI rule matching layer.

In [ ]:
WINDOW_LEN = 3

window_df = create_sliding_windows(
    df_feat,
    group_col='ID_PAZIENTE',
    time_col='DATAORA_ACCETTAZIONE',
    window_len=WINDOW_LEN,
    numeric_only=False,   # CRITICAL: preserves text columns for Italian rule matching
)

print(f'Window matrix shape : {window_df.shape}')
print(f'Unique patients     : {window_df["ID_PAZIENTE"].nunique()}')
print(f'Windows per patient : {len(window_df) / window_df["ID_PAZIENTE"].nunique():.1f} (avg)')
print(f'\nSample columns (first 12):')
print([c for c in window_df.columns[:12]])

## Step 4 – Define the Normal Baseline (Training Split)

**Unsupervised strategy**: patients with `num_visits_90d <= 2` at each window
anchor point are treated as the "normal" population on which the KDE density
is fitted.  High-frequency visitors are withheld (they form the anomalous tail
the detector should flag).

Adjust this threshold if your dataset's visit distribution is different.

In [ ]:
NORMAL_THRESHOLD_VISITS = 2   # windows where num_visits_90d_t <= this are "normal"

# The most recent visit in each window carries the _t suffix.
normal_mask = window_df['num_visits_90d_t'] <= NORMAL_THRESHOLD_VISITS

X_train_df = window_df[normal_mask].copy()
X_all_df   = window_df.copy()

# Extract the numeric sub-matrix for KDE fitting.
numeric_cols = X_train_df.select_dtypes(include=[np.number]).columns.tolist()

X_train_numeric = X_train_df[numeric_cols].values

print(f'Total windows          : {len(window_df)}')
print(f'Normal training windows: {len(X_train_df)}  ({100*len(X_train_df)/len(window_df):.1f}%)')
print(f'Anomalous/held-out     : {len(window_df) - len(X_train_df)}')
print(f'Numeric feature columns: {len(numeric_cols)}')

## Step 5 – Fit the KDE Anomaly Detector

In [ ]:
KDE_BANDWIDTH = 1.0   # Increase for smoother density; decrease for sharper.

detector = EthicalKDEAnomalyDetector(
    bandwidth=KDE_BANDWIDTH,
    kernel='gaussian',
    prior_penalty=1.5,
)

detector.fit(X_train_numeric)
print(detector)

## Step 6 – Score All Windows & Inspect the Anomaly Signal

In [ ]:
# Pass the full mixed-type DataFrame; the detector internally selects numeric columns.
anomaly_scores = detector.get_anomaly_signal(X_all_df)
window_df['anomaly_score'] = anomaly_scores.values

print('Anomaly score distribution:')
print(window_df['anomaly_score'].describe().round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Score distribution.
axes[0].hist(window_df['anomaly_score'], bins=40, color='#2c3e50', edgecolor='white')
axes[0].set_xlabel('Anomaly Score (-log P_KDE)')
axes[0].set_ylabel('Count')
axes[0].set_title('Score Distribution – All Windows')

# Normal vs. held-out comparison.
scores_normal = window_df.loc[normal_mask, 'anomaly_score']
scores_suspect = window_df.loc[~normal_mask, 'anomaly_score']
axes[1].hist(scores_normal,  bins=30, alpha=0.6, label='Normal (train)',  color='#27ae60')
axes[1].hist(scores_suspect, bins=30, alpha=0.6, label='High-freq (held)', color='#e74c3c')
axes[1].set_xlabel('Anomaly Score')
axes[1].set_title('Normal vs. High-Frequency Visitors')
axes[1].legend()

plt.tight_layout()
plt.show()

# Suggest a candidate threshold at the 90th percentile of the normal scores.
candidate_threshold = float(np.percentile(scores_normal, 90))
print(f'\nCandidate alert threshold (90th pct of normal): {candidate_threshold:.4f}')

## Step 7 – Clinical Rules (Italian Symbolic AI Layer)

This step extracts structured clinical rules from the Italian PDF guidelines
("Quaderni della Regione Emilia-Romagna") using **`extract_rules_local_ollama()`**,
a fully local, privacy-preserving LLM extractor that runs via [Ollama](https://ollama.com).

No data leaves the machine. If the Ollama server is not running or no PDF is
found, the notebook automatically falls back to 6 curated Italian seed rules
(R001–R006: *fratture multiple*, *ustioni*, *ematomi*, *lesioni cutanee*,
*trauma cranico*, *ecchimosi*).

The rules are loaded into the detector via `set_prior_knowledge()` so that
any future call to `get_anomaly_signal()` automatically applies the additive
text-matching penalties.

In [ ]:
SEED_RULES = [
    {
        'rule_id': 'R001',
        'description': 'Multiple fractures in a child are a strong red flag for non-accidental injury.',
        'clinical_conditions': 'fratture multiple',
        'penalty_weight': 2.5,
    },
    {
        'rule_id': 'R002',
        'description': 'Skin lesions inconsistent with the reported mechanism of injury.',
        'clinical_conditions': 'lesioni cutanee',
        'penalty_weight': 1.8,
    },
    {
        'rule_id': 'R003',
        'description': 'Burns in a child, especially patterned or immersion burns, are highly suspicious.',
        'clinical_conditions': 'ustioni',
        'penalty_weight': 2.0,
    },
    {
        'rule_id': 'R004',
        'description': 'Diffuse bruising or haematomas inconsistent with developmental stage.',
        'clinical_conditions': 'ematomi',
        'penalty_weight': 1.5,
    },
    {
        'rule_id': 'R005',
        'description': 'Traumatic brain injury in a pre-verbal child without a clear accidental mechanism.',
        'clinical_conditions': 'trauma cranico',
        'penalty_weight': 2.2,
    },
    {
        'rule_id': 'R006',
        'description': 'Multiple ecchymoses in a child, especially on non-bony prominences.',
        'clinical_conditions': 'ecchimosi',
        'penalty_weight': 1.3,
    },
]

prior_rules = SEED_RULES  # always-available fallback

docs_dir = PROJECT_ROOT / 'docs'
pdf_candidates = [
    docs_dir / 'Quaderno_5_MA (1).pdf',
    docs_dir / 'Quaderno_6_MA (1).pdf',
    docs_dir / 'Fratture_e_Abuso_quad2_web (1).pdf',
]
target_pdf = next((p for p in pdf_candidates if p.exists()), None)

if target_pdf:
    print(f'PDF trovato: {target_pdf.name}. Estrazione regole via Ollama (llama3)...')
    from util.knowledge_extraction import extract_text_from_pdf, extract_rules_local_ollama
    try:
        pdf_text = extract_text_from_pdf(str(target_pdf))
        extracted = extract_rules_local_ollama(pdf_text, model_name='llama3')
        if extracted:
            prior_rules = extracted
            print(f'Estratte {len(prior_rules)} regole dal PDF.')
        else:
            print('ATTENZIONE: Ollama ha restituito 0 regole; uso le regole mock.')
    except Exception as exc:
        print(f'ATTENZIONE: Ollama non raggiungibile, uso le regole mock. ({exc})')
else:
    print('Nessun PDF trovato in docs/; uso le regole mock.')

# Load rules into the detector.
detector.set_prior_knowledge(prior_rules)
print(f'Regole caricate nel detector: {len(detector.prior_rules)}')
for r in detector.prior_rules:
    print(f"  [{r['rule_id']}] '{r['clinical_conditions']}' → +{r['penalty_weight']} penalty")


## Step 8 – Save Artifacts

In [ ]:
# 8a) Re-score with rules loaded (penalties are now applied).
anomaly_scores_with_rules = detector.get_anomaly_signal(X_all_df)
window_df["anomaly_score"] = anomaly_scores_with_rules.values

# 8b) Save window DataFrame using a PyArrow-extension-type-safe writer.
#
# ROOT CAUSE OF BOTH ArrowKeyError VARIANTS:
#   df.to_parquet() calls pa.Table.from_pandas() internally, which registers
#   pandas custom Arrow extension types (pandas.period, pandas.interval, etc.)
#   in PyArrow global process-level registry.  Two reciprocal failures:
#     - First run (some versions):  ArrowKeyError "arrow.py_extension_type not found"
#     - Re-run same kernel:         ArrowKeyError "pandas.period already defined"
#
# FIX: bypass from_pandas() entirely.  Build the Arrow Table column-by-column
# with pa.Table.from_arrays() and explicit primitive Arrow types.
# pa.timestamp, pa.float64, pa.int64, pa.string and pa.bool_ are all native
# Arrow types -- they do not require any extension type registration.
import pyarrow as pa
import pyarrow.parquet as pq

def _save_parquet_safe(df, path):
    df = df.copy()
    # Normalize object columns: plain str, empty string for nulls.
    for col in df.select_dtypes(include="object").columns:
        df[col] = (df[col].fillna("").astype(str)
                   .replace({"None": "", "nan": "", "NaT": ""}))
    # Coerce pandas extension types (ArrowDtype, StringDtype, PeriodDtype) to numpy.
    for col in df.columns:
        dtype = df[col].dtype
        if hasattr(dtype, "pyarrow_dtype") or isinstance(dtype, pd.StringDtype):
            df[col] = df[col].astype(object).fillna("").astype(str)
        if hasattr(dtype, "freq"):  # PeriodDtype carries a .freq attribute
            df[col] = df[col].astype(str)
    # Build Arrow arrays column-by-column with explicit primitive types.
    # from_arrays() never calls pandas-specific setup, so no extension
    # type registration is triggered -- safe for repeated kernel execution.
    arrays, fields = [], []
    for col in df.columns:
        s, dtype = df[col], df[col].dtype
        if pd.api.types.is_datetime64_any_dtype(dtype):
            # Store datetime as ISO 8601 string -- avoids any datetime extension
            # type lookup and round-trips cleanly via pd.to_datetime() on load.
            vals = s.dt.strftime("%Y-%m-%dT%H:%M:%S").fillna("").tolist()
            arr, fld = pa.array(vals, type=pa.string()), pa.field(col, pa.string())
        elif pd.api.types.is_bool_dtype(dtype):
            arr, fld = pa.array(s.tolist(), type=pa.bool_()), pa.field(col, pa.bool_())
        elif pd.api.types.is_integer_dtype(dtype):
            arr, fld = pa.array(s.tolist(), type=pa.int64()), pa.field(col, pa.int64())
        elif pd.api.types.is_float_dtype(dtype):
            arr, fld = pa.array(s.tolist(), type=pa.float64()), pa.field(col, pa.float64())
        else:
            arr, fld = pa.array(s.tolist(), type=pa.string()), pa.field(col, pa.string())
        arrays.append(arr)
        fields.append(fld)
    pq.write_table(pa.Table.from_arrays(arrays, schema=pa.schema(fields)), str(path))

parquet_path = DATA_DIR / "windows.parquet"
_save_parquet_safe(window_df, parquet_path)
print(f"Saved windows.parquet -> {parquet_path}  ({window_df.shape})")

# 8c) Save the fitted detector (includes scaler + KDE + prior_rules).
model_path = DATA_DIR / "kde_model.joblib"
joblib.dump(detector, model_path)
print(f"Saved kde_model.joblib -> {model_path}")

# 8d) Save the prior rules as JSON for the Streamlit app.
rules_path = DATA_DIR / "prior_rules.json"
with open(rules_path, "w", encoding="utf-8") as fh:
    json.dump(prior_rules, fh, ensure_ascii=False, indent=2)
print(f"Saved prior_rules.json -> {rules_path}  ({len(prior_rules)} rules)")

print("All artifacts saved successfully.")

In [ ]:
# Verification: reload and spot-check.
#
# pd.read_parquet() also calls pandas Arrow interop initialisation, which
# tries to re-register pandas.period in the same kernel where it was
# already registered by a prior execution of the write cell.
# Fix: read via pq.read_table() and extract columns with pa.Array.to_numpy()
# (numeric) or pa.Array.to_pylist() (strings). These are pure Arrow/Python
# operations -- no pandas<->Arrow metadata conversion, no extension type lookup.

def _read_parquet_safe(path):
    table = pq.read_table(str(path))
    data = {}
    for name in table.schema.names:
        col = table.column(name)
        ft  = table.schema.field(name).type
        if pa.types.is_integer(ft) or pa.types.is_floating(ft) or pa.types.is_boolean(ft):
            data[name] = col.to_numpy(zero_copy=False)
        else:
            data[name] = col.to_pylist()
    return pd.DataFrame(data)

detector_check = joblib.load(model_path)
windows_check  = _read_parquet_safe(parquet_path)

# DATAORA_ACCETTAZIONE was serialised as ISO 8601 string; restore to datetime.
if "DATAORA_ACCETTAZIONE" in windows_check.columns:
    windows_check["DATAORA_ACCETTAZIONE"] = pd.to_datetime(
        windows_check["DATAORA_ACCETTAZIONE"], errors="coerce"
    )

rules_check = json.loads(rules_path.read_text(encoding="utf-8"))

assert detector_check._fitted, "Reloaded detector is not fitted!"
assert "anomaly_score" in windows_check.columns, "anomaly_score column missing!"
assert len(rules_check) > 0, "No rules saved!"

score_col = windows_check["anomaly_score"]
print("Verification passed.")
print(f"  Detector   : {detector_check}")
print(f"  Windows    : {windows_check.shape}")
print(f"  Rules      : {len(rules_check)}")
print(f"  Score range: [{score_col.min():.4f}, {score_col.max():.4f}]")